In [1]:
# --- Setup & load ---------------------------------------------------------
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

DATA_DIR = Path("data")                       # folder next to this notebook
RAW_FILE = Path("..") / "data" / "Flight_Data_With_Weather_Final.csv"   # shared raw file at repo root
OUT_DIR = DATA_DIR / "clean"

AIRPORTS = ["ATL", "DFW"]                     # Challenge 5 scope
KEEP_ONLY_ATL_DFW = True                      # False = keep every airport in the file
WX_VARS = ["temp", "prcp", "wspd", "wpgt", "pres", "snow", "coco"]

raw = pd.read_csv(RAW_FILE, low_memory=False)
raw = raw.assign(flight_id=raw.index)         # row position in the raw file, to trace rows back
print("raw:", raw.shape)

raw: (148273, 124)


C:\Users\keith\AppData\Local\Temp\ipykernel_25936\1534873035.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  raw = raw.assign(flight_id=raw.index)         # row position in the raw file, to trace rows back


# Challenge 5 - data cleaning

1. Drop unnecessary columns
2. Sort rows in time order
3. Split into a `flights` table and a `weather` table

**Scope.** The brief says to use only ATL and DFW, departing from and arriving to. The next cell keeps every flight that touches either airport. Set `KEEP_ONLY_ATL_DFW = False` in the first cell to skip this.

In [2]:
df = raw.copy()
if KEEP_ONLY_ATL_DFW:
    df = df[df["ORIGIN"].isin(AIRPORTS) | df["DEST"].isin(AIRPORTS)].copy()
print("after airport filter:", df.shape)
print("origin in scope:", df["ORIGIN"].isin(AIRPORTS).sum(), "| dest in scope:", df["DEST"].isin(AIRPORTS).sum())

after airport filter: (25299, 124)
origin in scope: 16151 | dest in scope: 9461


## 0. Build proper timestamps (needed before the raw `hhmm` columns can be dropped)

The file stores times as `hhmm` integers plus a date string, all in **local airport time**. From these I build:

- `sched_dep`: scheduled departure (origin local time)
- `sched_arr`: scheduled arrival (destination local time)
- `dep_hour`, `arr_hour`: those times floored to the hour
- `arr_overnight`, `arr_wx_hour`: explained in the data quirk below. `dep_hour` and `arr_wx_hour` are the keys for joining weather back on.

The scheduled arrival may fall on the next day and time zones differ, so I recover the zone gap from the schedule itself:

$$\Delta_{tz} = \mathrm{round}_{60}\Big( (t_{arr} - t_{dep} - T_{elapsed}) \bmod 1440 \Big) \quad \text{(minutes, wrapped to } [-720, 720) \text{)}$$

$$\text{sched\_arr} = \text{sched\_dep} + T_{elapsed} + \Delta_{tz}$$

The weather hour H is UTC (Meteostat), the METAR at about H-1:52Z, so it is 4-6 h before the flight. Verified vs NOAA ISD 44/44, and vs the IEM METAR archive for all 17,713 hours (mean |temp diff| 0.03 C read as UTC, 3.48 C read as local). That is why `dep_hour` and `arr_wx_hour` are renamed to `wx_*_hour_utc_key` on save: they are join keys into a UTC table, not the flight's local hour.

**Data quirk found while checking.** For flights that land after midnight (`arr_overnight`), the destination weather was joined on the *departure* date plus the arrival clock hour, so it is the weather of the previous day. I checked this against ATL/DFW readings from other flights: the match is 100% at the departure-date hour and 0% at the true arrival hour. To keep the weather table correct, destination weather is stored under the time it was actually recorded (`arr_wx_hour`). The true arrival hour stays in `arr_hour`.

In [3]:
def hhmm_to_min(s):
    """hhmm integer (e.g. 1550, 2400) -> minutes since midnight."""
    return (s // 100) * 60 + (s % 100)

date = pd.to_datetime(df["FL_DATE"])
dep_min = hhmm_to_min(df["CRS_DEP_TIME"])
arr_min = hhmm_to_min(df["CRS_ARR_TIME"])
elapsed = df["CRS_ELAPSED_TIME"]

df["sched_dep"] = date + pd.to_timedelta(dep_min, unit="m")

gap = (arr_min - dep_min - elapsed) % 1440
gap = gap.where(gap < 720, gap - 1440)                  # wrap to [-720, 720)
tz_shift = (gap / 60).round() * 60
df["sched_arr"] = df["sched_dep"] + pd.to_timedelta(elapsed + tz_shift, unit="m")

df["dep_hour"] = df["sched_dep"].dt.floor("h")
df["arr_hour"] = df["sched_arr"].dt.floor("h")                       # true arrival hour
df["arr_overnight"] = df["sched_arr"].dt.normalize() != date          # lands on a later calendar day
# hour the file's destination weather really belongs to (departure date + arrival clock hour)
df["arr_wx_hour"] = date + pd.to_timedelta((df["CRS_ARR_TIME"] % 2400) // 100, unit="h")

# sanity checks against the raw clock columns
arr_clock = df["sched_arr"].dt.hour * 100 + df["sched_arr"].dt.minute
print("overnight arrivals:", int(df["arr_overnight"].sum()))
# expect 1 and 1: ATL->HOU on 2024-06-05 has inconsistent scheduled times in the source file
print("zone gap not a whole hour:", ((gap - tz_shift).abs() > 0).sum())
print("sched_arr clock != CRS_ARR_TIME:", (arr_clock != df["CRS_ARR_TIME"] % 2400).sum())
act = (df["sched_dep"] + pd.to_timedelta(df["DEP_DELAY"], unit="m")).dropna()
act_clock = act.dt.hour * 100 + act.dt.minute
print("sched_dep + DEP_DELAY clock != DEP_TIME:", (act_clock != df.loc[act.index, "DEP_TIME"] % 2400).sum())

overnight arrivals: 860
zone gap not a whole hour: 1
sched_arr clock != CRS_ARR_TIME: 1
sched_dep + DEP_DELAY clock != DEP_TIME: 0


C:\Users\keith\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pandas\core\arrays\timedeltas.py:1163: RuntimeWarning: invalid value encountered in cast
  int_data = data.astype(np.int64)


## 1. Drop unnecessary columns

Each group is listed separately so you can see why it goes and add or remove names easily.

In [4]:
wx_all = [f"{v}_{s}" for v in WX_VARS for s in ("ORIGIN", "DEST")]

drop_groups = {
    # diversion detail (>99.7% empty). DIVERTED itself is kept.
    "diversion detail": [c for c in df.columns if re.match(r"DIV\d?_", c)],
    # alternative IDs / labels for the same airport or airline
    "duplicate ids & labels": (
        [f"{p}_{s}" for p in ("ORIGIN", "DEST")
         for s in ("AIRPORT_ID", "AIRPORT_SEQ_ID", "CITY_MARKET_ID", "CITY_NAME",
                   "STATE_ABR", "STATE_FIPS", "STATE_NM", "WAC")]
        + ["OP_CARRIER_AIRLINE_ID", "OP_CARRIER"]          # OP_CARRIER == OP_UNIQUE_CARRIER
    ),
    # replaced by sched_dep / sched_arr (recompute parts with .dt when needed)
    "date & clock columns": [
        "YEAR", "QUARTER", "MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK", "FL_DATE",
        "DEP_TIME_BLK", "ARR_TIME_BLK",
        "CRS_DEP_TIME", "CRS_ARR_TIME", "DEP_TIME", "ARR_TIME", "WHEELS_OFF", "WHEELS_ON",
    ],
    # derivable from DEP_DELAY / ARR_DELAY / DISTANCE, or constant
    "redundant": [
        "DEP_DELAY_NEW", "ARR_DELAY_NEW", "DEP_DEL15", "ARR_DEL15",
        "DEP_DELAY_GROUP", "ARR_DELAY_GROUP", "DISTANCE_GROUP",
        "ACTUAL_ELAPSED_TIME", "FLIGHTS",
    ],
    # gate-return details (99% empty)
    "mostly empty": ["FIRST_DEP_TIME", "TOTAL_ADD_GTIME", "LONGEST_ADD_GTIME"],
    # weather columns with no information (constant)
    "constant weather": [c for c in wx_all if df[c].nunique() <= 1],
}

for name, cols in drop_groups.items():
    print(f"{name:24s} {len(cols):3d} columns")
to_drop = sorted({c for cols in drop_groups.values() for c in cols})
df = df.drop(columns=to_drop, errors="raise")
print("\nconstant weather dropped:", drop_groups["constant weather"])
print("remaining:", df.shape[1], "columns")
print(list(df.columns))

diversion detail          45 columns
duplicate ids & labels    18 columns
date & clock columns      14 columns
redundant                  9 columns
mostly empty               3 columns
constant weather           2 columns

constant weather dropped: ['snow_ORIGIN', 'snow_DEST']
remaining: 39 columns
['OP_UNIQUE_CARRIER', 'TAIL_NUM', 'OP_CARRIER_FL_NUM', 'ORIGIN', 'DEST', 'DEP_DELAY', 'TAXI_OUT', 'TAXI_IN', 'ARR_DELAY', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'temp_ORIGIN', 'prcp_ORIGIN', 'wspd_ORIGIN', 'wpgt_ORIGIN', 'pres_ORIGIN', 'coco_ORIGIN', 'temp_DEST', 'prcp_DEST', 'wspd_DEST', 'wpgt_DEST', 'pres_DEST', 'coco_DEST', 'flight_id', 'sched_dep', 'sched_arr', 'dep_hour', 'arr_hour', 'arr_overnight', 'arr_wx_hour']


## 2. Sort rows in time order

Sorted by scheduled departure (the information available before the flight), with stable tie-breakers so reruns give an identical order. Times are local to each flight's origin airport, so ATL (Eastern) and DFW (Central) rows are not on one exact UTC timeline. This does not matter for per-airport hourly modelling.

In [5]:
df = (
    df.sort_values(["sched_dep", "ORIGIN", "DEST", "OP_UNIQUE_CARRIER", "OP_CARRIER_FL_NUM"],
                   kind="stable")
      .reset_index(drop=True)
)
assert df["sched_dep"].is_monotonic_increasing
print(df["sched_dep"].min(), "->", df["sched_dep"].max())

2023-01-01 05:00:00 -> 2025-07-31 21:01:00


## 3. Separate weather and flights

Weather is stored twice per flight (`*_ORIGIN`, `*_DEST`). I stack both sides into one long table keyed by `(airport, time)`, drop duplicates and check that no `(airport, time)` has two different readings. Flights keep `ORIGIN`, `DEST`, `dep_hour` and `arr_wx_hour`, so the two tables can be re-joined exactly, and lagged for the 3 to 12 hour forecast.

In [6]:
wx_vars = [v for v in WX_VARS if f"{v}_ORIGIN" in df.columns]      # snow dropped above

def side_weather(side, time_col):
    mask = df[side].isin(AIRPORTS) if KEEP_ONLY_ATL_DFW else pd.Series(True, index=df.index)
    out = df.loc[mask, [side, time_col] + [f"{v}_{side}" for v in wx_vars]].copy()
    out.columns = ["airport", "time"] + wx_vars
    return out

weather = pd.concat([side_weather("ORIGIN", "dep_hour"), side_weather("DEST", "arr_wx_hour")],
                    ignore_index=True).drop_duplicates()

conflicts = weather.duplicated(["airport", "time"]).sum()
assert conflicts == 0, f"{conflicts} (airport, time) pairs have conflicting readings"

weather = weather.sort_values(["time", "airport"]).reset_index(drop=True)
flights = df.drop(columns=[f"{v}_{s}" for v in wx_vars for s in ("ORIGIN", "DEST")])

print("flights:", flights.shape, "| weather:", weather.shape)
flights.head()

flights: (25299, 27) | weather: (17721, 8)


,OP_UNIQUE_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN,DEST,DEP_DELAY,TAXI_OUT,TAXI_IN,ARR_DELAY,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,flight_id,sched_dep,sched_arr,dep_hour,arr_hour,arr_overnight,arr_wx_hour
0,DL,N6707A,756.0,ATL,JFK,38.0,23.0,33.0,50.0,0.0,NaN,0.0,133.0,89.0,760.0,38.0,0.0,12.0,0.0,0.0,1664,2023-01-01 05:00:00,2023-01-01 07:13:00,2023-01-01 05:00:00,2023-01-01 07:00:00,False,2023-01-01 07:00:00
1,DL,N342NW,447.0,LGA,ATL,13.0,9.0,8.0,-2.0,0.0,NaN,0.0,156.0,124.0,762.0,NaN,NaN,NaN,NaN,NaN,1701,2023-01-01 06:00:00,2023-01-01 08:36:00,2023-01-01 06:00:00,2023-01-01 08:00:00,False,2023-01-01 08:00:00
2,NK,N919NK,207.0,DFW,LAS,-5.0,14.0,4.0,-11.0,0.0,NaN,0.0,178.0,154.0,1055.0,NaN,NaN,NaN,NaN,NaN,2165,2023-01-01 06:05:00,2023-01-01 07:03:00,2023-01-01 06:00:00,2023-01-01 07:00:00,False,2023-01-01 07:00:00
3,DL,N332DN,1631.0,BOS,ATL,13.0,15.0,7.0,-15.0,0.0,NaN,0.0,183.0,133.0,946.0,NaN,NaN,NaN,NaN,NaN,1668,2023-01-01 09:30:00,2023-01-01 12:33:00,2023-01-01 09:00:00,2023-01-01 12:00:00,False,2023-01-01 12:00:00
4,DL,N339NW,2743.0,ATL,SAV,-6.0,27.0,4.0,-5.0,0.0,NaN,0.0,66.0,36.0,214.0,NaN,NaN,NaN,NaN,NaN,1653,2023-01-01 10:10:00,2023-01-01 11:16:00,2023-01-01 10:00:00,2023-01-01 11:00:00,False,2023-01-01 11:00:00


In [7]:
weather.head()

,airport,time,temp,prcp,wspd,wpgt,pres,coco
0,ATL,2023-01-01 05:00:00,13.3,0.0,13.0,0.0,1017.2,5.0
1,DFW,2023-01-01 06:00:00,15.0,0.0,22.3,0.0,1009.1,2.0
2,ATL,2023-01-01 08:00:00,12.8,0.0,0.0,0.0,1017.2,5.0
3,ATL,2023-01-01 10:00:00,12.2,0.0,0.0,0.0,1017.6,5.0
4,DFW,2023-01-01 10:00:00,11.7,0.0,9.4,0.0,1009.4,2.0


### Checks

In [8]:
# 1) No information lost: every in-scope flight end can be re-joined to its weather
wx_key = weather.set_index(["airport", "time"])
dep_ok = flights.set_index(["ORIGIN", "dep_hour"]).index[flights["ORIGIN"].isin(AIRPORTS)]
arr_ok = flights.set_index(["DEST", "arr_wx_hour"]).index[flights["DEST"].isin(AIRPORTS)]
print("origin weather missing after split:", (~dep_ok.isin(wx_key.index)).sum())
print("dest weather missing after split:  ", (~arr_ok.isin(wx_key.index)).sum())

# 1b) Overnight arrivals: is there a weather row at the TRUE arrival hour?
ov = flights[flights["arr_overnight"] & flights["DEST"].isin(AIRPORTS)]
true_ok = ov.set_index(["DEST", "arr_hour"]).index.isin(wx_key.index)
print(f"overnight arrivals into scope: {len(ov)}; true-arrival-hour weather available for {true_ok.sum()}")

# 2) Both tables are time-ordered
assert flights["sched_dep"].is_monotonic_increasing and weather["time"].is_monotonic_increasing

# 3) The weather table only has hours with at least one scheduled flight, so it has gaps
span = (weather["time"].max() - weather["time"].min()).total_seconds() / 3600 + 1
cov = weather.groupby("airport").size().to_frame("hours_present")
cov["hours_in_span"] = int(span)
cov["coverage"] = (cov["hours_present"] / cov["hours_in_span"]).round(3)
print(cov)

# 4) Nulls
print("\nflights nulls (>0 only):")
print(flights.isna().sum().loc[lambda s: s > 0])
print("weather nulls:", int(weather.isna().sum().sum()))

origin weather missing after split: 0
dest weather missing after split:   0
overnight arrivals into scope: 386; true-arrival-hour weather available for 57
         hours_present  hours_in_span  coverage
airport                                        
ATL               9411          22625     0.416
DFW               8310          22625     0.367

flights nulls (>0 only):
TAIL_NUM                  28
DEP_DELAY                336
TAXI_OUT                 345
TAXI_IN                  357
ARR_DELAY                423
CANCELLATION_CODE      24954
AIR_TIME                 423
CARRIER_DELAY          19616
WEATHER_DELAY          19616
NAS_DELAY              19616
SECURITY_DELAY         19616
LATE_AIRCRAFT_DELAY    19616
dtype: int64
weather nulls: 0


## Save

In [ ]:
weather = weather.rename(columns={"time": "obs_hour_utc"})
flights = flights.rename(columns={"dep_hour": "wx_origin_hour_utc_key", "arr_wx_hour": "wx_dest_hour_utc_key"})
flights["dep_hour_local"] = flights["sched_dep"].dt.floor("h")

POST_EVENT = ["DEP_DELAY", "ARR_DELAY", "TAXI_OUT", "TAXI_IN", "AIR_TIME", "CANCELLED", "CANCELLATION_CODE",
              "DIVERTED", "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"]
flights = flights.rename(columns={c: f"post_{c}" for c in POST_EVENT})   # never use post_* or y_* as inputs
canc, div = flights["post_CANCELLED"] == 1, flights["post_DIVERTED"] == 1
flights["y_arr_severe"] = ((flights["post_ARR_DELAY"] > 45) | canc | div).astype(int)
flights["y_dep_severe"] = ((flights["post_DEP_DELAY"] > 45) | canc).astype(int)

weather["coco"] = weather["coco"].where(weather["coco"].between(1, 27))
weather["wpgt"] = np.nan          # unreliable in source; use IEM gust_kt
weather["wspd_kt"] = weather["wspd"] / 1.852

OUT_DIR.mkdir(parents=True, exist_ok=True)
flights.to_csv(OUT_DIR / "flights_clean.csv", index=False)
weather.to_csv(OUT_DIR / "weather_clean.csv", index=False)
print("saved to", OUT_DIR.resolve())
# reload later with: pd.read_csv(..., parse_dates=["sched_dep", "sched_arr", "wx_origin_hour_utc_key", "arr_hour"])